In [2]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
ws_config={
    "subscription_id": "7b1b43ca-4b64-43cf-9446-edb35a04d7d1",
    "resource_group": "AMLWS06",
    "workspace_name": "azuremlwest"
}

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

Found the config file in: .\config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [39]:
%%writefile train_hpt.py
import argparse
import pandas as pd 
import numpy as np 
#import mltable
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
#import mlflow
#from azure.ai.ml import command 
#from azure.ai.ml import MLClient
#from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from sklearn.preprocessing import LabelEncoder
import pickle
import mlflow
import os 
parser=argparse.ArgumentParser()
parser.add_argument('--regularization', type=float, dest='reg_rate', default=0.01)
parser.add_argument('--training_data', type=str, dest='train')
parser.add_argument('--solver', type=str, dest='solver')
args=parser.parse_args()

path=os.path.join(args.train, 'diabetes.csv')
df=pd.read_csv(path)
df.dropna(inplace=True)

y = df[['Outcome']]

x = df[['Pregnancies', 'Glucose', 'BloodPressure',
        'SkinThickness', 'Insulin', 'BMI',
        'DiabetesPedigreeFunction', 'Age']]
LE=LabelEncoder()
y=LE.fit_transform(y)
x_train, x_test, y_train, y_test=train_test_split(x,y, test_size=0.3)

model=LogisticRegression(solver=args.solver,C=1/args.reg_rate)
model.fit(x_train,y_train)
os.makedirs("outputs",exist_ok=True)
with open("outputs/model.pkl", "wb") as f:
        pickle.dump(model, f)
    #mlflow.sklearn.log_model(model, "model")
#mlflow.log_artifact("model.pkl")
y_hat=model.predict(x_test)

from sklearn.metrics import accuracy_score
acc=accuracy_score(y_hat, y_test)
mlflow.log_metric(acc)
print(f"Accuracy is {acc}")
#mlflow.log_metric("accuracy", acc)






Overwriting train_hpt.py


In [36]:
data_asset=ml_client.data.get('diabetes_mltable', version='4')
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.constants import AssetTypes, InputOutputModes

job=command(
code="./",
command="python train_hpt.py --solver ${{inputs.solver}} --regularization ${{inputs.regularization}} --training_data ${{inputs.data}}",
 inputs={
            "data": Input(path=data_asset.id,
                type=AssetTypes.URI_FOLDER,
                mode=InputOutputModes.RO_MOUNT
            ),
            "regularization":0.01,
            "solver":"lbfgs"
        },
        environment='custom-env-scikit:2',
        #"azureml:AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
        compute='satyakebakshi951'
    
)

In [37]:
from azure.ai.ml.sweep import Choice, BayesianSamplingAlgorithm, BanditPolicy, TruncationSelectionPolicy

command_job_for_sweep=job(
regularization=Choice(values=[0.01,0.1,1]),
solver=Choice(values=['liblinear','saga','lbfgs'])

)

sweep_job=command_job_for_sweep.sweep(
    compute="satyakebakshi951",
    sampling_algorithm=BayesianSamplingAlgorithm(),
    early_termination_policy=TruncationSelectionPolicy(evaluation_interval=2, truncation_percentage=20, delay_evaluation=2),
    primary_metric="Accuracy",
    goal="Maximize"
)

sweep_job.experiment_name="sweep_hypersweep_diabetes"
sweep_job.set_limits(max_total_trials=4, max_concurrent_trials=4, timeout=3600)

In [38]:
ml_client.create_or_update(sweep_job)

Experiment,Name,Type,Status,Details Page
sweep_hypersweep_diabetes,teal_floor_nl065gh7b9,sweep,Running,Link to Azure Machine Learning studio
